# OpenSOC-AI — Evaluation

Evaluates **baseline TinyLlama** vs **fine-tuned OpenSOC-AI** on the held-out eval set.

**Prerequisites:**
- Cell 1 deps installed + runtime restarted
- `soc_eval.json` on Google Drive
- `opensoc-adapters/` folder on Google Drive (from training)

**Metrics:**
- Threat Classification Accuracy
- Severity Accuracy
- MITRE ID Accuracy

In [ ]:
# ============================================================
# CELL 4: Evaluate Baseline vs Fine-Tuned
# Saves eval_results.json to Google Drive
# ============================================================

import os, json, re, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

DRIVE_ROOT   = '/content/drive/MyDrive'
EVAL_PATH    = f'{DRIVE_ROOT}/soc_eval.json'
ADAPTER_PATH = f'{DRIVE_ROOT}/opensoc-adapters'
EVAL_OUT     = f'{DRIVE_ROOT}/eval_results.json'
BASE_MODEL   = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

with open(EVAL_PATH) as f: eval_data = json.load(f)
print(f'✅ Eval dataset: {len(eval_data)} examples')

def extract_fields(text):
    fields = {}
    for key in ['THREAT_TYPE', 'SEVERITY', 'MITRE_ID']:
        match = re.search(rf'{key}[:\s]+([^\n,]+)', text, re.IGNORECASE)
        if match: fields[key] = match.group(1).strip().upper()
    return fields

def run_inference(model, tokenizer, example, max_new_tokens=256):
    prompt = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

def evaluate_model(model, tokenizer, eval_data, label):
    results = {'threat_correct':0, 'severity_correct':0, 'mitre_correct':0, 'total':len(eval_data), 'predictions':[]}
    for i, ex in enumerate(eval_data):
        pred_text = run_inference(model, tokenizer, ex)
        pred = extract_fields(pred_text)
        true = extract_fields(ex['output'])
        t = pred.get('THREAT_TYPE') == true.get('THREAT_TYPE')
        s = pred.get('SEVERITY')    == true.get('SEVERITY')
        m = pred.get('MITRE_ID')    == true.get('MITRE_ID')
        results['threat_correct']   += int(t)
        results['severity_correct'] += int(s)
        results['mitre_correct']    += int(m)
        results['predictions'].append({'index':i, 'predicted':pred, 'ground_truth':true, 'threat_correct':t, 'severity_correct':s})
        if (i+1) % 10 == 0: print(f'  [{label}] {i+1}/{len(eval_data)} done...')
    n = results['total']
    results['threat_accuracy']   = round(results['threat_correct']   / n, 4)
    results['severity_accuracy'] = round(results['severity_correct'] / n, 4)
    results['mitre_accuracy']    = round(results['mitre_correct']    / n, 4)
    return results

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
tokenizer  = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# Baseline
print('\n📊 Evaluating BASELINE...')
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map='auto')
base_model.eval()
baseline = evaluate_model(base_model, tokenizer, eval_data, 'BASELINE')
del base_model; torch.cuda.empty_cache()

# Fine-tuned
print('\n📊 Evaluating FINE-TUNED...')
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map='auto')
ft_model   = PeftModel.from_pretrained(base_model, ADAPTER_PATH)  # ✅ correct reload
ft_model.eval()
finetuned  = evaluate_model(ft_model, tokenizer, eval_data, 'FINE-TUNED')
del ft_model, base_model; torch.cuda.empty_cache()

# Summary
improvements = {
    'threat_accuracy_delta':   round(finetuned['threat_accuracy']   - baseline['threat_accuracy'],   4),
    'severity_accuracy_delta': round(finetuned['severity_accuracy'] - baseline['severity_accuracy'], 4),
    'mitre_accuracy_delta':    round(finetuned['mitre_accuracy']    - baseline['mitre_accuracy'],    4),
}
eval_results = {
    'model': 'TinyLlama-1.1B + LoRA (opensoc-adapters)',
    'eval_examples': len(eval_data),
    'baseline':    {k:v for k,v in baseline.items()  if k != 'predictions'},
    'finetuned':   {k:v for k,v in finetuned.items() if k != 'predictions'},
    'improvements': improvements,
    'per_example':  finetuned['predictions'],
}
with open(EVAL_OUT, 'w') as f: json.dump(eval_results, f, indent=2)

print('\n' + '='*52)
print('           EVALUATION SUMMARY')
print('='*52)
print(f'{"Metric":<26} {"Baseline":>9} {"Fine-tuned":>10} {"Delta":>7}')
print('-'*52)
for m in ['threat_accuracy','severity_accuracy','mitre_accuracy']:
    b = baseline[m]; f = finetuned[m]; d = improvements[m.replace('accuracy','accuracy_delta')]
    print(f'{m:<26} {b:>9.1%} {f:>10.1%} {"+" if d>=0 else ""}{d:.1%}:>7}')
print('='*52)
print(f'\n✅ Results saved to: {EVAL_OUT}')